# 01 · Exploración de la API del BOE y construcción del corpus

**Proyecto:** RAG legal informativo sobre legislación consolidada del BOE (TFG).
**Objetivo del notebook:** documentar el recorrido y las decisiones tomadas hasta ahora —
de la API del BOE hasta el corpus de chunks listo para indexar— mostrando los artefactos
reales generados por el pipeline.

> Aviso: los textos consolidados del BOE tienen **carácter informativo** y no valor jurídico
> oficial. Este sistema no es asesoramiento jurídico.

Principios del proyecto: empezar pequeño y cerrar el ciclo completo; priorizar la estructura
jurídica frente al chunking por longitud; trazabilidad y evaluación por encima de escala.
Las capas están desacopladas (cliente ≠ parser ≠ chunking) y la lógica reutilizable vive en
`src/`, de modo que este notebook solo orquesta y muestra.

In [ ]:
# Setup: localizar la raíz del repo (para importar `src` y leer artefactos) sin depender
# del directorio desde el que se abra el notebook.
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "pyproject.toml").is_file():
            return cand
    return p


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def load_json(rel_path: str):
    """Carga un artefacto JSON relativo a la raíz del repo (o avisa si falta)."""
    path = REPO_ROOT / rel_path
    if not path.is_file():
        print(f"[falta artefacto] {rel_path} -> genéralo con los scripts del README.")
        return None
    return json.loads(path.read_text(encoding="utf-8"))


NORM = "BOE-A-2015-10565"  # norma de referencia (Ley 39/2015)
print("Repo:", REPO_ROOT)

## 1. La API del BOE: legislación consolidada

Usamos la **API de legislación consolidada** (`/datosabiertos/api/legislacion-consolidada`),
no scraping HTML ni endpoints antiguos. Cada norma se identifica por su id BOE
(`BOE-A-YYYY-NNNNN`) y se expone por endpoints separados. Todos devuelven el mismo envoltorio:
`response > status (code/text) + data`. El cliente valida `status/code == 200` antes de usar
`data`.

Decisión registrada en `docs/fuentes_y_licencias.md`: estos son los 6 endpoints que
descargamos en crudo.

In [ ]:
from src.boe.client import ENDPOINTS

display(pd.DataFrame([{"endpoint": k, "ruta_relativa": v} for k, v in ENDPOINTS.items()]))

## 2. Descarga raw + manifest reproducible

`BoeClient` descarga cada endpoint y guarda los bytes tal cual en
`data/raw/boe/<id>/<endpoint>.xml`, sin parsear. Además genera un **manifest** con `sha256` y
tamaño por fichero: permite repetir la descarga y verificar que el contenido no ha cambiado
(trazabilidad, clave del TFG). El XML crudo no se versiona; el manifest sí.

Comando: `uv run python scripts/download_boe_raw.py BOE-A-2015-10565`

In [ ]:
manifest = load_json(f"data/manifests/{NORM}.json")
if manifest:
    display(Markdown(f"**{manifest['norm_id']}** · descargado: `{manifest['downloaded_at']}`"))
    display(pd.DataFrame(manifest["files"])[["endpoint_name", "size_bytes", "sha256"]])

## 3. Modelo documental `boe_legal_document_v1`

El XML del BOE no se usa directamente: el parser lo convierte en un **modelo documental
propio** (capa de independencia). La unidad principal es el **bloque jurídico** (artículo,
disposición, preámbulo, firma), no un trozo por longitud, porque es la unidad de cita y
recuperación nativa. Cada bloque conserva sus **versiones** y selecciona la **vigente**
(`latest_version`) por la `fecha_actualizacion` del bloque en `indice.xml` (coincidencia
exacta + única + máxima `fecha_publicacion`), **no** por el orden de las `<version>` en el XML;
guarda su **jerarquía** a 6 niveles (libro/título/capítulo/sección/subsección/anexo) y un objeto
**retrieval** con `citation_label` y `source_url`. Contrato completo en
`docs/modelo_documental.md`.

Comando: `uv run python scripts/parse_boe_raw.py BOE-A-2015-10565`

In [ ]:
doc = load_json(f"data/processed/documents/{NORM}.json")
if doc:
    m = doc["metadata"]
    display(Markdown(f"### {m['title']}"))
    resumen = {
        "short_title": m["short_title"],
        "rango": (m["rank"] or {}).get("label"),
        "ámbito": (m["scope"] or {}).get("label"),
        "estado_consolidacion": (m["consolidation_status"] or {}).get("label"),
        "fecha_publicacion": m["publication_date"],
        "fecha_vigencia": m["effective_date"],
    }
    display(pd.Series(resumen, name="metadata"))

In [ ]:
# Ejemplo de bloque con varias versiones (art?culo 9, modificado tras 2015).
if doc:
    block = next(b for b in doc["blocks"] if b["block_id"] == "a9")
    print("block_type:", block["block_type"], "| jerarqu?a:", block["hierarchy"])
    print("full_title:", block["full_title"])
    display(pd.DataFrame(block["versions"]))
    latest = block["latest_version"]
    print("\ntexto vigente (primeros 280 car.):\n", latest["text"][:280], "...")
    print("\nnotas de modificaci?n (fuera del texto):", len(latest["modification_notes"]))

### Decisiones del parser v0

- Usa los ficheros **separados** (`metadatos`, `analisis`, `indice`, `texto`), no `full.xml`
  (que no incluye el índice). `metadata_eli.xml` no se parsea en v0 (RDF redundante).
- `latest_version` lleva el texto; `versions[]` guarda solo metadatos (no se duplica el
  histórico). El MVP indexa solo la **versión vigente**, seleccionada por la `fecha_actualizacion`
  del índice (no por orden XML): si no resuelve de forma exacta y única, el bloque va a
  **cuarentena** (no indexable, sin chunks) y bloquea el avance a embeddings.
- Las notas de modificación (`nota_pie`) se separan a `modification_notes` y **no** entran en
  el texto recuperable. `quality_checks` valida conteos índice/texto y la resolución temporal.

In [ ]:
if doc:
    display(pd.Series(doc["quality_checks"], name="quality_checks"))

## 4. Chunking jurídico parent-child

Sobre el modelo documental, el chunker crea unidades recuperables **por párrafos** (no por
cortes de caracteres), manteniendo el bloque como **padre**: cada chunk lleva `parent_id` y
`parent_text` (el bloque completo). Solo se chunkean bloques indexables (artículos,
disposiciones, preámbulo); se excluyen encabezados y firma. `retrieval_text` antepone contexto
(norma + jerarquía + título) sin alterar el texto legal. Esquema: `boe_legal_chunks_v1`.

Comando: `uv run python scripts/chunk_boe_document.py BOE-A-2015-10565`

In [ ]:
chunks_doc = load_json(f"data/processed/chunks/{NORM}.json")
if chunks_doc:
    display(pd.Series(chunks_doc["chunking_strategy"], name="estrategia"))
    display(pd.Series(chunks_doc["quality_checks"], name="quality_checks"))
    ch = chunks_doc["chunks"][0]
    print("chunk_id:", ch["chunk_id"], "| parent_id:", ch["parent_id"])
    print("citation_label:", ch["metadata"]["citation_label"])
    print("retrieval_text (primeros 200 car.):\n", ch["retrieval_text"][:200])

## 5. Corpus semilla (10 normas) y verificación

Para probar el recuperador con variedad real ampliamos el MVP de 1 a **10 normas**
(`data/corpus/seed_corpus.json`). `scripts/build_corpus.py` descarga, **verifica** y procesa
cada una. Criterio de inclusión: **vigente** (no derogada, vigencia no agotada) **y**
`estado_consolidacion = Finalizado` **y** endpoints obligatorios disponibles. Las que no
cumplen se reportan y **no se sustituyen** automáticamente.

Comando: `uv run python scripts/build_corpus.py`

In [ ]:
from src.boe.corpus import load_seed_corpus

seed = load_seed_corpus(REPO_ROOT / "data" / "corpus" / "seed_corpus.json")
display(pd.DataFrame(seed))

In [ ]:
# Reporte de verificación (se genera al ejecutar build_corpus.py).
report = load_json("data/corpus/verification_report.json")
if report:
    rows = [
        {
            "norm_id": r["norm_id"],
            "rango": (r.get("rank") or {}).get("label"),
            "ámbito": (r.get("scope") or {}).get("label"),
            "estado": (r.get("consolidation_status") or {}).get("label"),
            "vigente": r.get("vigente"),
            "meets_criteria": r["meets_criteria"],
            "bloques": r.get("blocks_count"),
            "chunks": r.get("chunks_count"),
        }
        for r in report
    ]
    df = pd.DataFrame(rows)
    display(df)
    print("Cumplen criterios:", int(df["meets_criteria"].sum()), "de", len(df))

## 6. Próximos pasos

Con el corpus verificado y chunkeado, el siguiente paso es la **indexación**
(`src/indexing/`): embeddings + búsqueda semántica y BM25, partiendo de
`data/processed/chunks/<id>.json`. La evaluación del recuperador (Recall@k, MRR, artículo
correcto recuperado) se apoyará en `retrieval_text`, `parent_id`, `citation_label` y
`source_url` que ya viajan en cada chunk.